# 29 — Memory Optimization & Big Data

When data exceeds RAM, Pandas crashes. This notebook covers memory optimization
techniques: downcasting, pyarrow engine, categorical data, and chunking.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1_000_000  # 1 million rows

# Create a memory-heavy DataFrame
df = pd.DataFrame({
    'id':         np.arange(n),
    'amount':     np.random.uniform(10, 1000, n),
    'age':        np.random.randint(18, 80, n),
    'category':   np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], n),
    'is_active':  np.random.choice([True, False], n),
    'date_str':   pd.date_range('2020-01-01', periods=n, freq='min').astype(str)
})

def mem_usage(d):
    return f"{d.memory_usage(deep=True).sum() / 1024**2:.1f} MB"

print(f"Original Memory: {mem_usage(df)}")
print(df.dtypes)

## 1. Downcasting Numeric Types

- `int64` allows huge numbers, but `int8` (up to 127) or `int16` is often enough.
- `float64` has massive precision, `float32` is usually sufficient for ML/analytics.

In [ ]:
df_opt = df.copy()

# downcast float64 to float32
df_opt['amount'] = pd.to_numeric(df_opt['amount'], downcast='float')

# downcast int64 to lowest possible int
df_opt['age'] = pd.to_numeric(df_opt['age'], downcast='integer')
df_opt['id'] = pd.to_numeric(df_opt['id'], downcast='integer')

print(f"After downcasting: {mem_usage(df_opt)}")
print(df_opt[['id', 'amount', 'age']].dtypes)

## 2. Strings to Category

Object dtypes are pointers to Python strings (huge overhead).
Categories store the unique strings once and map them to integers.

In [ ]:
print("Category col before:", df_opt['category'].memory_usage(deep=True) // 1024**2, "MB")

df_opt['category'] = df_opt['category'].astype('category')

print("Category col after: ", df_opt['category'].memory_usage(deep=True) // 1024**2, "MB")
print(f"Total memory now: {mem_usage(df_opt)}")

## 3. String to Datetime

Dates stored as strings are objects. Convert them to `datetime64`.

In [ ]:
print("Date col before:", df_opt['date_str'].memory_usage(deep=True) // 1024**2, "MB")

df_opt['date_str'] = pd.to_datetime(df_opt['date_str'])

print("Date col after: ", df_opt['date_str'].memory_usage(deep=True) // 1024**2, "MB")
print(f"Total memory now: {mem_usage(df_opt)}")

## 4. PyArrow Engine (Pandas 2.0+)

Pandas 2.0 introduced PyArrow backing. It's much faster and memory efficient, especially for strings.

In [ ]:
# Convert string to PyArrow string type
df_arrow = df.copy()

# PyArrow string type (string[pyarrow] or string)
df_arrow['category'] = df_arrow['category'].astype('string[pyarrow]')
df_arrow['date_str'] = df_arrow['date_str'].astype('string[pyarrow]')

print(f"Arrow backed memory: {mem_usage(df_arrow)}")

# Note: When reading CSV, you can use: pd.read_csv('file.csv', dtype_backend='pyarrow')

## 5. Chunking (Reading Large CSVs)

If a file is larger than RAM, read it in chunks, process, and aggregate.

In [ ]:
# Simulating chunked reading
# import io
# chunk_iter = pd.read_csv('massive_file.csv', chunksize=100_000)
# 
# total_revenue = 0
# for chunk in chunk_iter:
#     # Process each chunk independently
#     clean_chunk = chunk.dropna(subset=['amount'])
#     total_revenue += clean_chunk['amount'].sum()
#
# print(total_revenue)